In [1]:
from src import *
import numpy as np
from tqdm import tqdm
from time import sleep

In [6]:
sampling_rate = 20 #Hz
freq_list = np.array([0.24, 0.245, 0.25, 0.255, 0.26])

pic_time = (1 / (sampling_rate * freq_list) / 2 * 1e6).astype(int)


A = 5
samples = 50
repeat = 200

interval = 50e-3
exposure_time = 500e-6 #500 us

measurement = 'SPADE'

In [7]:
with EasyDcam() as dcam:
    with EasyALP4() as alp:
        for i, picture_time in enumerate(pic_time):
            print(f'({i}): ')
            ground_truth = np.round(freq_list[i], 5)

            alp.ez_load_seq([alp.ez_single_pixel(0), alp.ez_single_pixel(A)], picture_time)
            dcam.ez_exposure_time(exposure_time)
            dcam.ez_triggersource_masterpluse(samples, interval)

            if measurement.upper() == 'SPADE':
                dcam.ez_roi(**SPADE.ROI)
            elif measurement.upper() == 'DI':
                dcam.ez_roi(**DI.ROI)

            raw, timestamp = [], []
            for _ in tqdm(range(repeat)):
                dcam.buf_alloc(samples)
                dcam.cap_snapshot()

                alp.Run()
                sleep(1e-6)
                dcam.cap_firetrigger()

                dcam.ez_wait_capture()

                dcam.cap_stop()
                alp.Halt()

                raw_, timestamp_ = [], []
                for frame in range(samples):
                    framedata_ = dcam.ez_read_buf(frame)
                    raw_.append(framedata_[0])
                    timestamp_.append(framedata_[1])

                dcam.buf_release()

                raw.append(raw_)
                timestamp.append(timestamp_)

            raw = np.array(raw)
            timestamp = np.array(timestamp)

            metadata = MetaData(measurement, ground_truth, A*DMD.PIXEL_SIZE/2, timestamp)
            est = FrequencyEstmation(np.array(raw), measurement, metadata)

            est.savez(f'./__temp__/{measurement.lower()}_{ground_truth}.npz')
            np.save(f'./__raw__/{measurement.lower()}_{ground_truth}_raw.npy', raw)


Loading library: c:\users\zzbn\Desktop\freqest\src\api/x64/alp4395.dll
DMD found, resolution = 1024 x 768.
(0): 


100%|██████████| 200/200 [09:05<00:00,  2.73s/it]


exited
exited


TypeError: _save_dispatcher() missing 1 required positional argument: 'arr'